# 3D Motion Capture — Google Colab Runner

End-to-end pipeline that converts a monocular video into a 3D animated skeleton and exports BVH, NPZ, and JSON animation files.

**Quick start (sample videos included — no setup needed):**
1. Run Cell 2 — detects environment. Drive mount is **optional**, skip it unless your video is on Drive.
2. Run Cell 3 — installs dependencies. If NumPy needs downgrade, runtime auto-restarts; re-run all cells after.
3. Run Cell 4 — clones the repo.
4. Run Cell 5 — pick a sample video by index, or set `VIDEO_PATH` to your own file.
5. Run Cell 6 — runs the full pipeline.
6. Run Cell 7 — view results summary.
7. Run Cell 8 — download output files.

> **GPU:** Enable via *Runtime → Change runtime type → T4 GPU* for faster processing.
> **Own video:** Set `VIDEO_PATH` in Cell 5 to a path on Drive (mount Drive first) or upload via the Files panel.


In [ ]:
# ── Cell 2: Environment detection ───────────────────────────────────────────
import os
import sys

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print(f'Running in Colab: {IN_COLAB}')

# ── Google Drive mount (OPTIONAL) ────────────────────────────────────────────
# Only needed if your video lives on Drive.
# Sample videos in sample_videos/ work without Drive — leave this False.
MOUNT_DRIVE = False

if IN_COLAB and MOUNT_DRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        print('Google Drive mounted at /content/drive')
    except Exception as _e:
        print(f'Drive mount failed: {_e}')
        print('Continuing without Drive — use sample_videos/ or upload a file directly.')
elif IN_COLAB:
    print('Drive mount skipped (MOUNT_DRIVE=False). sample_videos/ used by default.')
else:
    print('Local environment — Drive mount skipped.')


In [ ]:
# ── Cell 3: Install dependencies ────────────────────────────────────────────
import subprocess
import sys
import importlib
import importlib.metadata

PY_VER = sys.version_info[:2]
print(f'Python {PY_VER[0]}.{PY_VER[1]}')

if PY_VER >= (3, 13):
    print('Python 3.13 detected — using numpy 2.x (required for 3.13).')
    print('mediapipe support on Python 3.13 is limited; pipeline will use fallbacks if needed.')

def _pkg_version(pkg: str):
    try:
        v = importlib.metadata.version(pkg)
        return tuple(int(x) for x in v.split('.')[:3])
    except Exception:
        return None

def _pkg_installed(name: str) -> bool:
    # catch ALL exceptions — AttributeError from numpy version mismatch is common
    try:
        importlib.import_module(name)
        return True
    except Exception:
        return False

def _pip_install(spec: str, label: str = '') -> bool:
    r = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '--quiet', spec],
        capture_output=True,
    )
    if r.returncode != 0:
        print(f'  WARNING: install failed for {label or spec}')
        print(r.stderr.decode()[-600:])
        return False
    return True

# ── NumPy: only downgrade if on Python <=3.12 AND numpy>=2.0 ────────────────
# Python 3.13 REQUIRES numpy>=2.0 — do NOT downgrade on 3.13+
_np_ver = _pkg_version('numpy')
if PY_VER <= (3, 12) and (_np_ver is None or _np_ver >= (2, 0, 0)):
    print(f'NumPy {_np_ver} detected on Python {PY_VER[0]}.{PY_VER[1]} — downgrading to <2.0 ...')
    _pip_install('numpy>=1.24,<2.0', 'numpy<2.0')
    print('NumPy downgraded — restarting runtime ...')
    import os
    os.kill(os.getpid(), 9)

print(f'NumPy {_pkg_version("numpy")} OK')

# ── Core packages ────────────────────────────────────────────────────────────
# mediapipe>=0.10.14 supports numpy 2.x; may not install on Python 3.13 — that is OK
CORE_DEPS = [
    ('mediapipe', 'mediapipe>=0.10.14'),
    ('cv2',       'opencv-python>=4.8'),
    ('scipy',     'scipy>=1.11'),
    ('yaml',      'PyYAML>=6.0'),
    ('tqdm',      'tqdm>=4.66'),
    ('h5py',      'h5py>=3.9'),
]

failed = []
for _mod, _spec in CORE_DEPS:
    if not _pkg_installed(_mod):
        print(f'Installing {_spec} ...')
        if not _pip_install(_spec, _mod):
            failed.append(_mod)
            print(f'  Skipping {_mod} — pipeline will use fallbacks.')

if failed:
    print(f'\nPackages unavailable (fallbacks will be used): {failed}')
else:
    print('All core dependencies ready.')


In [ ]:
# ── Cell 4: Clone / locate project ──────────────────────────────────────────
import os
import sys
import subprocess
from pathlib import Path

REPO_URL     = 'https://github.com/typist1/mocap.git'
PROJECT_ROOT = ''   # local path override (non-Colab only)

# ── Resolve project root ─────────────────────────────────────────────────────
if IN_COLAB:
    _clone_dir = Path('/content/mocap3d')
    if not _clone_dir.exists():
        print(f'Cloning {REPO_URL} ...')
        subprocess.check_call(['git', 'clone', '--depth', '1', REPO_URL, str(_clone_dir)])
        print('Clone complete.')
    else:
        print('Repo present — fetching latest ...')
        try:
            subprocess.check_call(['git', '-C', str(_clone_dir), 'fetch', '--depth=1', 'origin'])
            subprocess.check_call(['git', '-C', str(_clone_dir), 'reset', '--hard', 'origin/main'])
            print('Updated to latest.')
        except subprocess.CalledProcessError as _e:
            print(f'Git update failed ({_e}) — using existing clone.')
    _root = _clone_dir
else:
    if not PROJECT_ROOT:
        _nb_dir = Path(os.path.abspath(''))
        _root = _nb_dir.parent if (_nb_dir / 'src').exists() or (_nb_dir.parent / 'src').exists() else _nb_dir
        if not (_root / 'src').exists():
            raise FileNotFoundError(
                'Cannot locate project root automatically. Set PROJECT_ROOT explicitly.'
            )
        print(f'Auto-detected project root: {_root}')
    else:
        _root = Path(PROJECT_ROOT)
        if not _root.exists():
            raise FileNotFoundError(f'PROJECT_ROOT does not exist: {_root}')

PROJECT_ROOT = str(_root)
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
    print(f'Added to sys.path: {PROJECT_ROOT}')

# ── Install from requirements.txt ────────────────────────────────────────────
_req = _root / 'requirements.txt'
if _req.exists():
    print(f'Installing from {_req} ...')
    subprocess.check_call(
        [sys.executable, '-m', 'pip', 'install', '--quiet', '-r', str(_req)]
    )
    print('Requirements installed.')

print(f'Project root: {PROJECT_ROOT}')


In [ ]:
# ── Cell 5: Configuration ────────────────────────────────────────────────────
from pathlib import Path

# ── Pick a video from sample_videos/ ────────────────────────────────────────
_sample_dir = Path(PROJECT_ROOT) / 'sample_videos'
_videos = sorted(
    [f for f in _sample_dir.iterdir() if f.suffix.lower() in ('.mp4', '.mov', '.avi', '.mkv', '.webm')]
) if _sample_dir.exists() else []

if _videos:
    print('Available videos in sample_videos/:')
    for i, v in enumerate(_videos):
        print(f'  [{i}] {v.name}')
    print()

# ── Set VIDEO_INDEX to pick, or override VIDEO_PATH directly ─────────────────
VIDEO_INDEX = 0        # index from the list above
VIDEO_PATH  = ''       # set this directly to override VIDEO_INDEX

USE_GPU          = True   # Set False to force CPU inference
TARGET_FPS       = 30.0   # Output animation frame rate
SMOOTHING_CUTOFF = 6.0    # Butterworth cutoff Hz (lower = smoother, higher = snappier)
EXPORT_FORMATS   = ['bvh', 'npz', 'json']  # any subset
OUTPUT_DIR       = ''    # leave empty to auto-create next to video
CONFIG_FILE      = ''    # optional YAML override

# ── Resolve paths ────────────────────────────────────────────────────────────
if not VIDEO_PATH:
    if not _videos:
        raise FileNotFoundError(f'No videos found in {_sample_dir}. Add .mp4/.mov files there or set VIDEO_PATH directly.')
    if VIDEO_INDEX >= len(_videos):
        raise IndexError(f'VIDEO_INDEX {VIDEO_INDEX} out of range (0–{len(_videos)-1}).')
    VIDEO_PATH = _videos[VIDEO_INDEX]
else:
    VIDEO_PATH = Path(VIDEO_PATH)

if not VIDEO_PATH.exists():
    raise FileNotFoundError(f'Video not found: {VIDEO_PATH}')

if not OUTPUT_DIR:
    OUTPUT_DIR = VIDEO_PATH.parent / (VIDEO_PATH.stem + '_mocap_out')

OUTPUT_DIR = Path(OUTPUT_DIR)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Video      : {VIDEO_PATH}')
print(f'Output dir : {OUTPUT_DIR}')
print(f'GPU        : {USE_GPU}')
print(f'Target FPS : {TARGET_FPS}')
print(f'Cutoff Hz  : {SMOOTHING_CUTOFF}')
print(f'Formats    : {EXPORT_FORMATS}')

In [ ]:
# ── Cell 6: Run pipeline ─────────────────────────────────────────────────────
import time
import logging
from pathlib import Path
from src.config.settings import load_settings
from src.pipeline import MocapPipeline

logging.basicConfig(
    level=logging.INFO,
    format='%(levelname)s %(name)s: %(message)s',
    force=True,  # reconfigure on cell re-run
)

# Load settings from project config (absolute path works in Colab + local)
_config_path = Path(PROJECT_ROOT) / 'configs' / 'default.yaml'
settings = load_settings(_config_path if _config_path.exists() else None)

# Apply notebook-level overrides
settings.video.target_fps             = TARGET_FPS
settings.filtering.butterworth_cutoff = SMOOTHING_CUTOFF
settings.animation.export_formats     = EXPORT_FORMATS
settings.animation.output_dir         = str(OUTPUT_DIR)
settings.device                       = 'auto' if USE_GPU else 'cpu'

# Run
pipeline = MocapPipeline(settings)
print(f'Starting pipeline on: {VIDEO_PATH.name}')
t0 = time.perf_counter()

result = pipeline.process(video_path=VIDEO_PATH, output_dir=OUTPUT_DIR)

elapsed = time.perf_counter() - t0
print(f'\nPipeline finished in {elapsed:.1f}s')


In [ ]:
# ── Cell 7: View results ─────────────────────────────────────────────────────
from IPython.display import display, Video, HTML

# ── Summary ─────────────────────────────────────────────────────────────────
print('=== Processing Summary ===')
print(f'  Source         : {result.source_video.name}')
print(f'  Frames         : {result.frame_count}')
print(f'  Detection rate : {result.detection_rate * 100:.1f}%')
print(f'  Elapsed        : {result.processing_time_sec:.2f}s')
if result.frame_count and result.processing_time_sec:
    print(f'  Throughput     : {result.frame_count / result.processing_time_sec:.1f} fps')

if result.output_files:
    print('\n  Output files:')
    for fmt, path in sorted(result.output_files.items()):
        size_kb = path.stat().st_size / 1024 if path.exists() else 0
        print(f'    [{fmt:>5}]  {path.name}  ({size_kb:.1f} KB)')

if result.metrics is not None:
    print(f'\n  Metrics: {result.metrics}')

# ── Preview video ────────────────────────────────────────────────────────────
_preview = result.output_files.get('preview')
if _preview is None:
    # Try common naming convention
    _candidate = OUTPUT_DIR / (VIDEO_PATH.stem + '_preview.mp4')
    _preview = _candidate if _candidate.exists() else None

if _preview and _preview.exists():
    print(f'\nPreview: {_preview}')
    display(Video(str(_preview), embed=True, width=640))
else:
    print('\nNo preview video available.')


In [ ]:
# ── Cell 8: Download results ─────────────────────────────────────────────────
if IN_COLAB:
    from google.colab import files as colab_files

    if not result.output_files:
        print('No output files to download.')
    else:
        print('Downloading output files...')
        for fmt, path in sorted(result.output_files.items()):
            if path.exists():
                print(f'  Downloading {path.name} ...')
                colab_files.download(str(path))
            else:
                print(f'  Skipping {path.name} (file not found)')
        print('Download complete.')
else:
    print('Not running in Colab — files are already at:')
    for fmt, path in sorted(result.output_files.items()):
        print(f'  {path}')
